# Silver - Clientes CRM

Padronização cadastral dos clientes, limpeza de chaves e formatação de strings.

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
sistema = 'case'
table_name = 'crm_clientes_export'
output_table_name = 'crm_clientes'
input_path = f"{var_bronze}/{sistema}/{table_name}/data"
output_path_data = f"{var_silver}/{sistema}/{output_table_name}/data"
table_name_schema = f'{var_environment}.{var_silver_schema}.{sistema}_{output_table_name}'

In [ ]:
from pyspark.sql.functions import col, regexp_replace, lower, trim, to_date

df_bronze = spark.read.format("delta").load(input_path)

df_clean = (
    df_bronze
    .withColumn("documento_limpo", regexp_replace(col("documento"), r"[.\-/]", ""))
    .select(
        col("id_cliente").cast("integer").alias("id_cliente"),
        trim(col("nome")).cast("string").alias("nome_cliente"),
        lower(trim(col("email"))).cast("string").alias("email_cliente"),
        col("documento_limpo").cast("string").alias("documento_cliente"),
        col("tipo_documento").cast("string").alias("tipo_documento_cliente"),
        col("estado").cast("string").alias("uf_cliente"),
        to_date(col("data_cadastro")).alias("data_cadastro_cliente")
    )
    .filter(col("id_cliente").isNotNull())
    .dropDuplicates(["id_cliente"])
)

In [ ]:
process_data(
    df_write=df_clean,
    tipo_carga='delta',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['uf_cliente'],
    chave_upsert='id_cliente'
)